In [9]:
import gc, sys
from pathlib import Path
import numpy as np
import pandas as pd
from itertools import combinations
import torch
from google.colab import drive, runtime

IN_COLAB = True
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/IV Project-Apr 1, 2026')
sys.path.insert(0, str(ROOT))

from src.paths import CLEAN_DATA
from src.benchmark import analytic_benchmark
from src.helper import make_run_dir
from src.fully_connected_colab import (
    compute_batch_size,
    detect_device,
    prepare_gpu_data,
    save_colab_run,
    train_feature_sweep,
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Load data

In [10]:
DATA_SET = 'rand_A'

df_train = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_train.parquet'))
df_val   = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_val.parquet'))
df_test  = pd.read_parquet(CLEAN_DATA / (DATA_SET + '_test.parquet'))


OUTPUT_DIR = make_run_dir()

## GPU auto-detect

In [11]:
CFG = detect_device()
DEVICE = CFG['DEVICE']
AMP_DTYPE = CFG['POLICY']
USE_AMP = (AMP_DTYPE != torch.float32)

A100-80GB  |  VRAM: 85 GB  |  MAX_BATCH=65,536  |  dtype=torch.bfloat16


## Hyperparameters

In [12]:
SEED = 42
MAX_EPOCHS = 100
PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.3
WARMUP_EPOCHS = 5
NEURONS = 80
HIDDEN_LAYERS = 3

BATCH_SIZE = compute_batch_size(len(df_train), CFG['MAX_BATCH'])
BASE_LR = 1e-3
BASE_BATCH = 4096
INIT_LR = BASE_LR * (BATCH_SIZE / BASE_BATCH) ** 0.5

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

steps_per_epoch = len(df_train) // BATCH_SIZE
print(f'MAX_BATCH={CFG["MAX_BATCH"]:,}  adaptive BATCH_SIZE={BATCH_SIZE:,}  '
      f'INIT_LR={INIT_LR:.6f}  n_train={len(df_train):,}  '
      f'steps/epoch~{steps_per_epoch}')
print(f'MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  '
      f'WARMUP={WARMUP_EPOCHS} epochs')

MAX_BATCH=65,536  adaptive BATCH_SIZE=32,768  INIT_LR=0.002828  n_train=2,652,048  steps/epoch~80
MAX_EPOCHS=100  PATIENCE=30  WARMUP=5 epochs


## Feature definitions

In [13]:
from itertools import combinations

FEATURES_3 = ['delta', 'T', 'spy_ret']
EXTRA_FEATURES = ['vix_lag', 'vix_mom_lag', 'vix_mom', 'gamma', 'theta', 'vega', 'rho']
ALL_FEATURES = FEATURES_3 + EXTRA_FEATURES
TARGET = 'd_iv'

# Generate ALL possible combinations of EXTRA_FEATURES (0 to 7)
feature_combos = []
base_name = '+'.join(FEATURES_3)
feature_combos.append((base_name, FEATURES_3.copy()))  # Base case

for r in range(1, len(EXTRA_FEATURES) + 1):
    for combo in combinations(EXTRA_FEATURES, r):
        feats = FEATURES_3 + list(combo)
        name = '+'.join(feats)
        feature_combos.append((name, feats))

# Calculate breakdown for all sizes
n_plus_0 = 1
n_plus_1 = len(list(combinations(EXTRA_FEATURES, 1)))
n_plus_2 = len(list(combinations(EXTRA_FEATURES, 2)))
n_plus_3 = len(list(combinations(EXTRA_FEATURES, 3)))
n_plus_4 = len(list(combinations(EXTRA_FEATURES, 4)))
n_plus_5 = len(list(combinations(EXTRA_FEATURES, 5)))
n_plus_6 = len(list(combinations(EXTRA_FEATURES, 6)))
n_plus_7 = len(list(combinations(EXTRA_FEATURES, 7)))

print(f'Total feature combinations: {len(feature_combos)}')
print('  Base (3F):  1')
print(f'  3F + 1:     {n_plus_1}')
print(f'  3F + 2:     {n_plus_2}')
print(f'  3F + 3:     {n_plus_3}')
print(f'  3F + 4:     {n_plus_4}')
print(f'  3F + 5:     {n_plus_5}')
print(f'  3F + 6:     {n_plus_6}')
print(f'  3F + 7:     {n_plus_7}')

Total feature combinations: 128
  Base (3F):  1
  3F + 1:     7
  3F + 2:     21
  3F + 3:     35
  3F + 4:     35
  3F + 5:     21
  3F + 6:     7
  3F + 7:     1


## Pre-allocate on GPU

In [14]:
gpu_data = prepare_gpu_data(df_train, df_val, df_test, ALL_FEATURES, TARGET, DEVICE)
COL_IDX = gpu_data['col_idx']

Data on GPU  |  VRAM used: 0.81 GB / 85 GB  |  Free: 84.3 GB
Train: 2,652,048  Val: 757,728  Test: 378,865  Features: 10


## Analytic benchmark

In [15]:
hw = analytic_benchmark(df_train, df_val, df_test, target=TARGET)

Analytic Benchmark
SSE = 115.3004  RMSE = 0.017445
Coefficients: a = -0.137147, b = -0.079071, c = -0.056296


## Train all feature combinations

In [16]:
train_kw = dict(
    Xtr=gpu_data['Xtr'],
    Xva=gpu_data['Xva'],
    Xte=gpu_data['Xte'],
    ytr=gpu_data['ytr'],
    yva=gpu_data['yva'],
    y_test=gpu_data['y_test'],
    hw_sse=hw['sse'],
    all_feature_names=ALL_FEATURES,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    use_amp=USE_AMP,
    nan_mask_tr=gpu_data['nan_mask_tr'],
    nan_mask_va=gpu_data['nan_mask_va'],
    nan_mask_ytr=gpu_data['nan_mask_ytr'],
    nan_mask_yva=gpu_data['nan_mask_yva'],
    seed=SEED,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    lr_patience=LR_PATIENCE,
    lr_factor=LR_FACTOR,
    init_lr=INIT_LR,
    warmup_epochs=WARMUP_EPOCHS,
    neurons=NEURONS,
    hidden_layers=HIDDEN_LAYERS,
)

sep = "=" * 70
print('\n' + sep)
print(f'  TRAINING {len(feature_combos)} MODELS')
print(f'  GPU: {CFG["GPU"]}  |  Batch: {BATCH_SIZE:,}  |  AMP: {USE_AMP}  |  Max epochs: {MAX_EPOCHS}')
print(sep + '\n')

all_results, elapsed_s = train_feature_sweep(
    feature_combos,
    col_idx=COL_IDX,
    train_kwargs=train_kw,
    print_every=25,
)

print(f'\nDone: {elapsed_s / 60:.1f} min for {len(all_results)} models '
      f'(avg {elapsed_s / max(len(all_results), 1):.1f}s/model)')


  TRAINING 128 MODELS
  GPU: A100-80GB  |  Batch: 32,768  |  AMP: True  |  Max epochs: 100

  [  1/128] delta+T+spy_ret                SSE=106.8820  Gain=+7.3%  ep=100  24.8s  elapsed=0.4min
  [ 25/128] delta+T+spy_ret+gamma+vega     SSE=98.3365  Gain=+14.7%  ep=100  19.0s  elapsed=8.1min
  [ 50/128] delta+T+spy_ret+vix_mom_lag+gamma+vega SSE=87.6425  Gain=+24.0%  ep=100  19.0s  elapsed=16.0min
  [ 75/128] delta+T+spy_ret+vix_lag+vix_mom+gamma+theta SSE=69.9920  Gain=+39.3%  ep=100  18.8s  elapsed=24.0min
  [100/128] delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+gamma+theta SSE=63.7371  Gain=+44.7%  ep=100  19.2s  elapsed=32.0min
  [125/128] delta+T+spy_ret+vix_lag+vix_mom_lag+gamma+theta+vega+rho SSE=70.5631  Gain=+38.8%  ep=100  18.8s  elapsed=39.9min
  [128/128] delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+gamma+theta+vega+rho SSE=62.8780  Gain=+45.5%  ep=100  18.8s  elapsed=40.9min

Done: 40.9 min for 128 models (avg 19.2s/model)


## Results summary

In [17]:
summary, df_results = save_colab_run(
    OUTPUT_DIR,
    y_test=gpu_data['y_test'],
    hw=hw,
    models=all_results,
)
summary

,Model,SSE,MSE,RMSE,MAE,MeanError,MedianAE,R2,Training_time,Gain_vs_Analytic,Gain_Incremental
0,Analytic,115.300354,0.000304,0.017445,0.006248,0.001539,0.002549,0.081748,None,None,None
1,delta+T+spy_ret,106.881950,0.000282,0.016796,0.007031,-0.000110,0.003620,0.148792,19.9s,7.30%,None
2,delta+T+spy_ret+vix_lag,113.535667,0.000300,0.017311,0.009037,0.000095,0.005690,0.095802,18.9s,1.53%,-6.23%
3,delta+T+spy_ret+vix_mom_lag,108.367325,0.000286,0.016912,0.008172,0.000746,0.004924,0.136963,19.2s,6.01%,4.55%
4,delta+T+spy_ret+vix_mom,105.543289,0.000279,0.016691,0.007959,0.002086,0.004703,0.159453,19.4s,8.46%,2.61%
...,...,...,...,...,...,...,...,...,...,...,...
125,delta+T+spy_ret+vix_lag+vix_mom_lag+gamma+thet...,70.563072,0.000186,0.013647,0.006403,-0.000107,0.003464,0.438036,18.8s,38.80%,4.82%
126,delta+T+spy_ret+vix_lag+vix_mom+gamma+theta+ve...,71.818771,0.000190,0.013768,0.006370,0.000428,0.003408,0.428035,19.4s,37.71%,-1.78%
127,delta+T+spy_ret+vix_mom_lag+vix_mom+gamma+thet...,72.589531,0.000192,0.013842,0.006245,-0.000556,0.003225,0.421897,18.9s,37.04%,-1.07%
128,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,62.877956,0.000166,0.012883,0.005951,-0.000925,0.003018,0.499240,18.8s,45.47%,13.38%


## Top 10 by Gain vs Hull-White

In [18]:
print('Top 10 feature combos by Gain vs Hull-White:\n')
top10 = df_results.head(10)[['combo_name', 'n_features', 'SSE', 'RMSE', 'Gain_vs_HW_%', 'training_time_s']].copy()
top10['Gain_vs_HW_%'] = top10['Gain_vs_HW_%'].round(2)
top10['RMSE'] = top10['RMSE'].round(6)
top10['SSE'] = top10['SSE'].round(4)
top10['training_time_s'] = top10['training_time_s'].round(1)
top10

Top 10 feature combos by Gain vs Hull-White:



,combo_name,n_features,SSE,RMSE,Gain_vs_HW_%,training_time_s
0,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,8,62.7529,0.012870,45.57,18.9
1,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,10,62.8780,0.012883,45.47,18.8
2,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+th...,8,63.2440,0.012920,45.15,18.9
3,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,8,63.7371,0.012970,44.72,19.1
4,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,9,64.5349,0.013051,44.03,19.1
5,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+ga...,8,65.8712,0.013186,42.87,18.7
6,delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+th...,8,65.9270,0.013191,42.82,19.2
7,delta+T+spy_ret+vix_mom_lag+vix_mom+gamma+thet...,8,68.3416,0.013431,40.73,19.1
8,delta+T+spy_ret+vix_lag+vix_mom+gamma+theta+rho,8,68.3457,0.013431,40.72,18.7
9,delta+T+spy_ret+vix_lag+vix_mom+theta+vega+rho,8,68.3468,0.013431,40.72,19.4


## Best per group (3F, +1, +2, +3)

In [19]:
for nf_label, n in [('3F (base)', 3), ('+1 (4F)', 4), ('+2 (5F)', 5), ('+3 (6F)', 6)]:
    sub = df_results[df_results['n_features'] == n]
    if len(sub) == 0:
        continue
    best = sub.iloc[0]
    print(f"{nf_label}: {best['combo_name']}")
    print(f"    SSE={best['SSE']:.4f}  RMSE={best['RMSE']:.6f}  Gain={best['Gain_vs_HW_%']:.2f}%\n")

3F (base): delta+T+spy_ret
    SSE=106.8820  RMSE=0.016796  Gain=7.30%

+1 (4F): delta+T+spy_ret+vix_mom
    SSE=105.5433  RMSE=0.016691  Gain=8.46%

+2 (5F): delta+T+spy_ret+vix_mom+theta
    SSE=80.5276  RMSE=0.014579  Gain=30.16%

+3 (6F): delta+T+spy_ret+vix_lag+vix_mom+gamma
    SSE=77.8974  RMSE=0.014339  Gain=32.44%



## Summary statistics

In [20]:
total_time_s = df_results['training_time_s'].sum()
print(f'Total training time: {total_time_s / 60:.1f} min ({total_time_s / 3600:.2f} hr)')
print(f'Models trained: {len(df_results)}')
print(f'Best overall: {df_results.iloc[0]["combo_name"]} (Gain={df_results.iloc[0]["Gain_vs_HW_%"]:.2f}%)')

Total training time: 40.8 min (0.68 hr)
Models trained: 128
Best overall: delta+T+spy_ret+vix_lag+vix_mom_lag+vix_mom+gamma+rho (Gain=45.57%)


## Cleanup

In [21]:
del all_results, gpu_data
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'Final VRAM: {(total - free) / 1e9:.2f} GB / {total / 1e9:.0f} GB')

print(f'Total training time: {total_time_s / 60:.1f} min for {len(df_results)} models')

if IN_COLAB:
    runtime.unassign()

Final VRAM: 0.96 GB / 85 GB
Total training time: 40.8 min for 128 models
